# 01 · Diseño en bloques completos al azar — DBCA (Python)

**Semana 2 — Bloqueo y factoriales.**

## Objetivos
- Ajustar y analizar un DBCA con un factor de tratamiento y un factor de bloque.
- Verificar supuestos y valorar la **eficiencia** del bloqueo.

## Paquetes
`pandas`, `numpy`, `matplotlib`, `scipy`, `statsmodels`.

> Teoría: [`../teoria/01-bloques-dbca.md`](../teoria/01-bloques-dbca.md).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

np.random.seed(2)

## 1. Los datos

Rendimiento de un proceso de **extrusión** según la **presión** (4 niveles: 8500–9100 psi, el tratamiento de interés), con **6 lotes** de materia prima como **bloque** (Montgomery, ej. 4.1). Un dato por celda presión×lote.

In [ ]:
df = pd.read_csv('../datos/rendimiento-extrusion.csv')
df['presion'] = df['presion'].astype('category')
df['lote'] = df['lote'].astype('category')
df.pivot(index='presion', columns='lote', values='rendimiento')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11,4))
df.boxplot(column='rendimiento', by='presion', ax=ax[0], grid=False)
ax[0].set_title('Por presión'); ax[0].set_xlabel('Presión')
df.boxplot(column='rendimiento', by='lote', ax=ax[1], grid=False)
ax[1].set_title('Por lote (bloque)'); ax[1].set_xlabel('Lote')
plt.suptitle(''); plt.tight_layout(); plt.show()

Los lotes muestran niveles distintos: bloquear por lote debería limpiar esa variabilidad del error.

## 2. ANOVA del DBCA

Modelo $y_{ij}=\mu+\tau_i+\beta_j+\varepsilon_{ij}$ (tratamiento = presión, bloque = lote, **sin interacción**).

In [ ]:
modelo = ols('rendimiento ~ C(presion) + C(lote)', data=df).fit()
tabla = sm.stats.anova_lm(modelo, typ=2)
print(tabla.round(4))

La **presión** es significativa ($p\approx0.002$): afecta el rendimiento. Los **lotes** también difieren ($p\approx0.006$), lo que confirma que **bloquear fue acertado** (de lo contrario esa variación habría inflado el error).

## 3. Verificación de supuestos

In [ ]:
resid = modelo.resid; ajust = modelo.fittedvalues
fig, ax = plt.subplots(1, 2, figsize=(11,4))
sm.qqplot(resid, line='s', ax=ax[0]); ax[0].set_title('Q-Q de residuales')
ax[1].scatter(ajust, resid); ax[1].axhline(0, color='gray', ls='--')
ax[1].set_xlabel('Ajustados'); ax[1].set_ylabel('Residual')
ax[1].set_title('Residuales vs. ajustados')
plt.tight_layout(); plt.show()
print('Shapiro:', stats.shapiro(resid))

El supuesto adicional del DBCA es la **aditividad** (no interacción tratamiento×bloque): con un dato por celda no es estimable, pero el patrón de residuales no sugiere problemas.

## 4. Comparación de presiones (Tukey)

In [ ]:
tuk = pairwise_tukeyhsd(df['rendimiento'], df['presion'])
print(tuk.summary())

## 5. Eficiencia relativa del bloqueo

Compara la precisión obtenida con la que habría dado un diseño completamente al azar.

In [ ]:
aov = sm.stats.anova_lm(modelo, typ=2)
a = df['presion'].nunique(); b = df['lote'].nunique()
CME = aov.loc['Residual', 'sum_sq'] / (a-1)/(b-1)
CMB = aov.loc['C(lote)', 'sum_sq'] / (b-1)
ER = ((b-1)*CMB + b*(a-1)*CME) / ((a*b-1)*CME)
print(f'Eficiencia relativa DBCA vs. DCA = {ER:.2f}')

$ER>1$ indica que el bloqueo **mejoró** la precisión: habríamos necesitado ~$ER$ veces más réplicas en un DCA para igualarla.

## 6. Conclusiones
- La presión de extrusión afecta el rendimiento ($p\approx0.002$).
- El bloqueo por lote fue **eficaz** (lotes significativos, $ER>1$).

> Equivalente en R: [`01-dbca_r.ipynb`](01-dbca_r.ipynb).